# Convo Accuracy Evaluation

Evaluates how accurately WhisperX transcribes the conversation task NS5 recordings by comparing against the human-annotated Praat word alignments (`word_info.csv`).

**Pipeline:**
1. Auto-discover patients with all required inputs (timing, word_info, whisperx JSON)
2. Parse convo WhisperX JSONs (`convo_comparison/{pt}/task_whisper/*.json`) into word-level DataFrames with the same context-window format as REBIRTH_2! transcripts
3. Build reference sentences from `anilu_comparison/{pt}/words/word_info.csv` (punctuation-based segmentation)
4. Match reference sentences to WhisperX hypothesis segments using WER + fuzzy string similarity (monotonic search)
5. Compute timing errors for aligned word pairs
6. Submit SBERT cosine similarity jobs via SLURM (GPU); notebook resumes once outputs are ready
7. Assemble per-patient and combined CSVs for Figure 2 plotting

**Outputs** (per patient): `convo_comparison/{pt}/accuracy/matches.csv`, `timings.csv`, `matches_sbert.csv`

**New patients are automatically picked up** as long as they have `anilu_comparison/{pt}/time/task_timing.json`, `anilu_comparison/{pt}/words/word_info.csv`, and at least one JSON in `convo_comparison/{pt}/task_whisper/`.

In [1]:
import bisect
import json
import re
import subprocess
from collections import deque
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein
from rapidfuzz.fuzz import token_set_ratio

In [2]:
# ── Config ─────────────────────────────────────────────────────────────────────
DATA_ROOT       = Path("/mnt/labworlds/Hayden/Hayden_Lab/speech_247")
ANILU_ROOT      = DATA_ROOT / "anilu_comparison"   # reference word_info.csv + task_timing.json
CONVO_COMP_ROOT = DATA_ROOT / "convo_comparison"   # output dir

PYTHON_BIN  = "/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3"
WORKER_PATH = Path("/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/convo_comparison/sbert_worker.py")
HF_HOME     = "/scratch/tahaismail424/hf"

PARTITION = "Universal"
GRES      = "gpu:1"
MEM       = "16G"
CONDA_ACTIVATE = "source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env"

# Matching params
CHEAP_THRESH        = 60    # token_set_ratio pre-filter (0..100)
WER_SIM_THRESH      = 0.50  # minimum WER similarity to accept a match
MIN_REF_TOKENS      = 2     # skip sentences with fewer raw normalized tokens (too_short)
MIN_CONTENT_TOKENS  = 2     # skip sentences with fewer content tokens (filler_only)
TIME_WINDOW         = 30.0  # seconds — search ±TIME_WINDOW around ref sentence midpoint
SIZE_SLACK          = 8     # hyp window size allowed to vary ± this many words vs ref length

## 1 — Patient discovery

In [3]:
def discover_patients() -> dict[str, dict]:
    """
    Returns {patient: info_dict} for every patient with all required inputs.
    Prefers word_info_timing_corrected.csv over word_info.csv when present —
    corrected CSVs are generated by align_ref_timing.ipynb for patients whose
    Praat timestamps don't align with the WhisperX audio timeline.
    """
    available: dict[str, dict] = {}
    skipped:   dict[str, list] = {}

    for pt_dir in sorted(ANILU_ROOT.iterdir()):
        if not pt_dir.is_dir():
            continue
        pt = pt_dir.name
        if not re.match(r"Y[A-Z]{2}$", pt):
            continue

        timing_json   = pt_dir / "time" / "task_timing.json"
        hyp_csv       = pt_dir / "whisperx_out" / "audio_transcript_df.csv"
        corrected_csv = pt_dir / "words" / "word_info_timing_corrected.csv"
        word_csv      = corrected_csv if corrected_csv.exists() else pt_dir / "words" / "word_info.csv"

        missing = []
        if not timing_json.exists(): missing.append("task_timing.json")
        if not word_csv.exists():    missing.append("word_info.csv")
        if not hyp_csv.exists():     missing.append("whisperx_out/audio_transcript_df.csv")

        if missing:
            skipped[pt] = missing
        else:
            available[pt] = {
                "timing_json":      timing_json,
                "word_csv":         word_csv,
                "hyp_csv":          hyp_csv,
                "out_dir":          CONVO_COMP_ROOT / pt / "accuracy",
                "timing_corrected": corrected_csv.exists(),
            }

    print(f"Available patients ({len(available)}): {sorted(available)}")
    corrected_pts = [p for p, v in available.items() if v["timing_corrected"]]
    if corrected_pts:
        print(f"  Using timing-corrected ref for: {corrected_pts}")
    if skipped:
        print(f"Skipped ({len(skipped)}):")
        for pt, reasons in sorted(skipped.items()):
            print(f"  {pt}: missing {reasons}")
    return available


patients = discover_patients()

Available patients (15): ['YEY', 'YEZ', 'YFA', 'YFC', 'YFF', 'YFG', 'YFI', 'YFK', 'YFM', 'YFN', 'YFP', 'YFR', 'YFS', 'YFU', 'YFV']
  Using timing-corrected ref for: ['YEZ', 'YFA']


## 2 — Parse convo WhisperX JSONs

In [4]:
def load_hyp_df(
    csv_path:  Path,
    utc_start: pd.Timestamp | None = None,
) -> pd.DataFrame:
    """
    Load anilu_comparison/{pt}/whisperx_out/audio_transcript_df.csv and
    normalise column names to match what build_hyp_segments / relative_timing_errors
    expect (word_start_s, word_end_s, word_score, segment_start_s, segment_end_s).
    """
    df = pd.read_csv(csv_path, index_col=0)

    df = df.rename(columns={
        "start_ts":       "word_start_s",
        "end_ts":         "word_end_s",
        "qscore":         "word_score",
        "segment_start":  "segment_start_s",
        "segment_end":    "segment_end_s",
    })

    if utc_start is not None:
        df["utc_word_start"] = utc_start + pd.to_timedelta(df["word_start_s"], unit="s")
        df["utc_word_end"]   = utc_start + pd.to_timedelta(df["word_end_s"],   unit="s")

    return df.reset_index(drop=True)

## 3 — Build reference from word_info.csv

In [5]:
def build_ref_df(
    word_csv:    Path,
    timing_json: Path,
) -> tuple[pd.DataFrame, pd.Timestamp]:
    """
    Load anilu word_info.csv, add UTC timestamps and sentence_id.

    onset/offset are milliseconds from the start of the convo recording.
    sentence_id increments after each sentence-terminal punctuation mark.
    Returns (df, utc_start).
    """
    with open(timing_json) as f:
        timing = json.load(f)
    utc_start = pd.Timestamp(timing["start_time_iso"])

    df = pd.read_csv(word_csv, index_col=0)
    df = df.sort_values("onset").reset_index(drop=True)

    # convert to seconds so timing is directly comparable to hyp
    df["word_start_s"] = df["onset"]  / 1_000.0
    df["word_end_s"]   = df["offset"] / 1_000.0

    df["utc_word_start"] = utc_start + pd.to_timedelta(df["onset"],  unit="ms")
    df["utc_word_end"]   = utc_start + pd.to_timedelta(df["offset"], unit="ms")

    sentence_id = 0
    ids = []
    for word in df["CollapsedWord"]:
        ids.append(sentence_id)
        if any(c in {"?", "!", "."} for c in str(word)):
            sentence_id += 1
    df["sentence_id"] = ids

    return df, utc_start

## 4 — WER / alignment utilities

In [6]:
_word_re = re.compile(r"[^\w']+", re.UNICODE)


def norm_word(w: str) -> str:
    return _word_re.sub("", w.lower().strip()) if w else ""


def norm_tokens(words: list[str]) -> list[str]:
    return [t for w in words if (t := norm_word(str(w)))]


def join_tokens(tokens: list[str]) -> str:
    return " ".join(tokens)


# Tokens that ASR (WhisperX) typically does not produce:
#   Praat inaudible placeholders, laughter annotations, filled pauses, backchannels.
FILLER_TOKENS: frozenset[str] = frozenset({
    "x",                                      # Praat placeholder for inaudible
    "haha", "hahaha", "hah", "ha",            # laughter
    "um", "umm", "uh", "uhh", "uhm",          # filled pauses
    "mmhmm", "mhm", "uhhuh",                  # backchannels
})


def build_content_mask(tokens: list[str]) -> list[int]:
    """
    Return indices (into `tokens`) of the content tokens: those not in
    FILLER_TOKENS, with consecutive identical tokens collapsed (stutter dedup).
    e.g. ["i","i","i","said"] → [0, 3]  (first "i" + "said")
    """
    mask: list[int] = []
    last: str | None = None
    for i, t in enumerate(tokens):
        if t in FILLER_TOKENS:
            continue
        if t == last:           # collapse stutter repeats
            continue
        mask.append(i)
        last = t
    return mask


def is_match(a: str, b: str) -> bool:
    if not a or not b:
        return False
    if a == b:
        return True
    dist = Levenshtein.distance(a, b)
    return dist <= (1 if min(len(a), len(b)) <= 4 else 2)


@dataclass
class AlignResult:
    pairs: list
    S: int
    D: int
    I: int
    N: int


def align_tokens(ref: list[str], hyp: list[str], match_fn=is_match) -> AlignResult:
    n, m = len(ref), len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    bt = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0], bt[i][0] = i, "D"
    for j in range(1, m + 1):
        dp[0][j], bt[0][j] = j, "I"

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = 0 if match_fn(ref[i - 1], hyp[j - 1]) else 1
            cands = [
                (dp[i-1][j-1] + sub, "M" if sub == 0 else "S"),
                (dp[i-1][j] + 1,     "D"),
                (dp[i][j-1] + 1,     "I"),
            ]
            dp[i][j], bt[i][j] = min(cands, key=lambda x: x[0])

    i, j = n, m
    pairs: list = []
    S = D = I = 0
    while i > 0 or j > 0:
        step = bt[i][j]
        if step in ("M", "S"):
            if step == "S":
                S += 1
            pairs.append((i - 1, j - 1, step == "M"))
            i -= 1; j -= 1
        elif step == "D":
            D += 1
            pairs.append((i - 1, None, False))
            i -= 1
        elif step == "I":
            I += 1
            pairs.append((None, j - 1, False))
            j -= 1
        else:
            break
    pairs.reverse()
    return AlignResult(pairs=pairs, S=S, D=D, I=I, N=n)


def wer_similarity(aln: AlignResult) -> float:
    if aln.N == 0:
        return float("nan")
    return max(0.0, 1.0 - (aln.S + aln.D + aln.I) / aln.N)

## 5 — Sentence / segment builders and monotonic matching

In [7]:
def build_ref_sentences(ref_df: pd.DataFrame) -> list[dict]:
    """
    Build one dict per sentence_id.  Carries full token list (ref_tokens) and
    content-filtered version (filler-stripped, stutter-deduped).
    ref_tokens is used for matching; content_tokens only for filler_only detection.
    """
    sents = []
    for sent_id, g in ref_df.groupby("sentence_id", sort=True):
        g = g.sort_values("word_start_s").reset_index(drop=True)
        tokens = norm_tokens(g["CollapsedWord"].tolist())

        content_mask = build_content_mask(tokens)
        content_toks = [tokens[i] for i in content_mask]

        sents.append({
            "sent_id":        sent_id,
            "ref_df":         g,
            "ref_tokens":     tokens,
            "ref_text":       join_tokens(tokens),
            "content_tokens": content_toks,
            "content_text":   join_tokens(content_toks),
        })
    return sents


def match_ref_to_hyp(
    ref_sents:          list[dict],
    hyp_df:             pd.DataFrame,
    cheap_thresh:       float = CHEAP_THRESH,
    wer_sim_thresh:     float = WER_SIM_THRESH,
    min_ref_tokens:     int   = MIN_REF_TOKENS,
    min_content_tokens: int   = MIN_CONTENT_TOKENS,
    time_window:        float = TIME_WINDOW,
    size_slack:         int   = SIZE_SLACK,
) -> list[dict]:
    """
    Match each ref sentence to its hyp counterpart using the shared audio timeline.

    Both Praat (ref) and WhisperX (hyp) annotate the same audio file, so their
    word_start_s values are in the same coordinate system.  For each ref sentence
    we search only the hyp words within ±time_window seconds of the ref midpoint.
    This anchors every sentence search independently — no cascade failures from a
    drifting monotonic pointer.

    Matching uses full ref_tokens (fillers become WER deletions, not spurious 
    short matches).  Filler-only sentences are classified via content_tokens and
    excluded from the match-rate denominator.
    """
    hyp_sorted = hyp_df.sort_values("word_start_s").reset_index(drop=True)

    # Build parallel arrays: normalised token, original row index, start time
    norm_entries: list[tuple[str, int, float]] = []
    for i, row in hyp_sorted.iterrows():
        t = norm_word(str(row["word"]))
        if t:
            norm_entries.append((t, int(i), float(row["word_start_s"])))

    hyp_norm   = [e[0] for e in norm_entries]   # normalised words
    hyp_row_ix = [e[1] for e in norm_entries]   # original hyp_sorted row index
    hyp_times  = [e[2] for e in norm_entries]   # word_start_s (sorted)

    matches: list[dict] = []

    for rs in ref_sents:
        ref_tokens     = rs["ref_tokens"]
        content_tokens = rs["content_tokens"]
        n = len(ref_tokens)

        if n < min_ref_tokens:
            matches.append({**rs, "matched": False, "reason": "too_short"})
            continue
        if len(content_tokens) < min_content_tokens:
            matches.append({**rs, "matched": False, "reason": "filler_only"})
            continue

        # Temporal anchor: midpoint of this ref sentence
        ref_g   = rs["ref_df"]
        ref_mid = (float(ref_g["word_start_s"].min()) + float(ref_g["word_end_s"].max())) / 2.0

        # Narrow hyp search to ±time_window around ref midpoint
        idx_lo = bisect.bisect_left(hyp_times,  ref_mid - time_window)
        idx_hi = bisect.bisect_right(hyp_times, ref_mid + time_window)

        best_wer: float       = -1.0
        best:     dict | None = None
        ref_text              = rs["ref_text"]

        for start in range(idx_lo, min(idx_hi, len(hyp_norm))):
            w_min = max(1, n - size_slack)
            w_max = n + size_slack

            for w_size in range(w_min, w_max + 1):
                end = start + w_size
                if end > len(hyp_norm):
                    break

                window   = hyp_norm[start : end]
                hyp_text = join_tokens(window)

                cs = float(token_set_ratio(ref_text, hyp_text))
                if cs < cheap_thresh:
                    continue

                aln = align_tokens(ref_tokens, window)
                ws  = wer_similarity(aln)

                if ws >= wer_sim_thresh and ws > best_wer:
                    best_wer = ws
                    row_idx  = hyp_row_ix[start : end]
                    hyp_sub  = hyp_sorted.loc[row_idx].reset_index(drop=True)
                    best = {
                        **rs,
                        "matched":        True,
                        "hyp_word_start": start,
                        "hyp_word_end":   end,
                        "segment_id":     (int(hyp_sub["segment_id"].iloc[0])
                                           if len(hyp_sub) else None),
                        "cheap_sim":      cs,
                        "wer_sim":        ws,
                        "S": aln.S, "D": aln.D, "I": aln.I,
                        "aln":            aln,
                        "hyp_df":         hyp_sub,
                        "hyp_tokens":     window,
                        "hyp_text":       hyp_text,
                        "segment_text":   (str(hyp_sub["segment_text"].iloc[0])
                                           if len(hyp_sub) else ""),
                    }

        if best is None:
            matches.append({**rs, "matched": False, "reason": "no_match"})
        else:
            matches.append(best)

    return matches

## 6 — Timing error

In [8]:
def relative_timing_errors(
    ref_df: pd.DataFrame,
    hyp_df: pd.DataFrame,
    aln:    AlignResult,
    start_col: str = "word_start_s",
    end_col:   str = "word_end_s",
) -> pd.DataFrame:
    """
    Relative midpoint timing error (seconds) for each matched word pair.
    Times are relative to sentence anchor so absolute recording-start drift cancels.
    """
    r_anchor = float(ref_df[start_col].min())
    h_anchor = float(hyp_df[start_col].min())

    def rel_mid(df: pd.DataFrame, idx: int, anchor: float) -> float:
        s = float(df.iloc[idx][start_col])
        e = float(df.iloc[idx][end_col])
        return (s + e) / 2.0 - anchor

    rows = []
    for ri, hi, match_bool in aln.pairs:
        if not match_bool or ri is None or hi is None:
            continue
        rows.append({
            "ref_i":           ri,
            "hyp_j":           hi,
            "ref_word":        ref_df.iloc[ri]["CollapsedWord"],
            "hyp_word":        hyp_df.iloc[hi]["word"],
            "delta_mid_rel_s": rel_mid(hyp_df, hi, h_anchor) - rel_mid(ref_df, ri, r_anchor),
        })
    return pd.DataFrame(rows)

## 7 — Load all patient data

In [9]:
pt_ref_dfs:    dict[str, pd.DataFrame]  = {}
pt_hyp_dfs:    dict[str, pd.DataFrame]  = {}
pt_utc_starts: dict[str, pd.Timestamp]  = {}

for pt, info in patients.items():
    ref_df, utc_start = build_ref_df(info["word_csv"], info["timing_json"])
    hyp_df = load_hyp_df(info["hyp_csv"], utc_start=utc_start)

    pt_ref_dfs[pt]    = ref_df
    pt_hyp_dfs[pt]    = hyp_df
    pt_utc_starts[pt] = utc_start

    print(
        f"{pt}: {len(ref_df)} ref words  |  "
        f"{len(hyp_df)} hyp words, {hyp_df['segment_id'].nunique()} segments"
    )

YEY: 5631 ref words  |  5114 hyp words, 707 segments
YEZ: 5279 ref words  |  4915 hyp words, 662 segments
YFA: 8260 ref words  |  6900 hyp words, 821 segments
YFC: 3544 ref words  |  2861 hyp words, 352 segments
YFF: 6089 ref words  |  4969 hyp words, 468 segments
YFG: 2140 ref words  |  1867 hyp words, 271 segments
YFI: 3772 ref words  |  7280 hyp words, 1052 segments
YFK: 13173 ref words  |  13402 hyp words, 1504 segments
YFM: 7744 ref words  |  13889 hyp words, 1662 segments
YFN: 3413 ref words  |  2574 hyp words, 373 segments
YFP: 9988 ref words  |  9578 hyp words, 903 segments
YFR: 7043 ref words  |  6949 hyp words, 893 segments
YFS: 8781 ref words  |  8340 hyp words, 1017 segments
YFU: 12395 ref words  |  11611 hyp words, 1498 segments
YFV: 11585 ref words  |  11678 hyp words, 1244 segments


## 8 — Run matching and timing for all patients

In [10]:
pt_matches: dict[str, pd.DataFrame] = {}
pt_timings: dict[str, pd.DataFrame] = {}

for pt, info in patients.items():
    ref_df = pt_ref_dfs[pt]
    hyp_df = pt_hyp_dfs[pt].copy()

    ref_sents = build_ref_sentences(ref_df)
    matches   = match_ref_to_hyp(ref_sents, hyp_df)

    sent_rows:  list[dict]         = []
    timing_dfs: list[pd.DataFrame] = []

    for m in matches:
        reason = ("matched" if m.get("matched") else m.get("reason", "no_match"))
        base: dict = {
            "sent_id":      m["sent_id"],
            "matched":      m.get("matched", False),
            "reason":       reason,
            "ref_n":        len(m["ref_tokens"]),
            "content_n":    len(m["content_tokens"]),
            "ref_text":     m["ref_text"],
            "content_text": m["content_text"],
        }
        if m.get("matched"):
            ref_g = m["ref_df"]
            hyp_g = m["hyp_df"]
            ref_seg_start = float(ref_g["word_start_s"].min())
            ref_seg_end   = float(ref_g["word_end_s"].max())
            ref_seg_mid   = (ref_seg_start + ref_seg_end) / 2.0
            hyp_seg_start = float(hyp_g["word_start_s"].min())
            hyp_seg_end   = float(hyp_g["word_end_s"].max())
            hyp_seg_mid   = (hyp_seg_start + hyp_seg_end) / 2.0

            base.update({
                "segment_id":        m.get("segment_id"),
                "cheap_sim":         m["cheap_sim"],
                "wer_sim":           m["wer_sim"],
                "S": m["S"], "D": m["D"], "I": m["I"],
                "hyp_n":             len(m["hyp_tokens"]),
                "hyp_text":          m["hyp_text"],
                # Segment-level timing
                "ref_seg_start_s":   ref_seg_start,
                "ref_seg_end_s":     ref_seg_end,
                "hyp_seg_start_s":   hyp_seg_start,
                "hyp_seg_end_s":     hyp_seg_end,
                "seg_start_drift_s": hyp_seg_start - ref_seg_start,   # + = hyp later
                "seg_mid_drift_s":   hyp_seg_mid   - ref_seg_mid,     # + = hyp later
            })
            td = relative_timing_errors(m["ref_df"], m["hyp_df"], m["aln"])
            td["sent_id"]    = m["sent_id"]
            td["segment_id"] = m.get("segment_id")
            timing_dfs.append(td)
        sent_rows.append(base)

    pt_matches[pt] = pd.DataFrame(sent_rows)
    pt_timings[pt] = pd.concat(timing_dfs, ignore_index=True) if timing_dfs else pd.DataFrame()

    df_pt       = pt_matches[pt]
    n_total     = len(df_pt)
    n_matched   = int(df_pt["matched"].sum())
    n_too_short = int((df_pt["reason"] == "too_short").sum())
    n_filler    = int((df_pt["reason"] == "filler_only").sum())
    n_content   = n_total - n_too_short - n_filler
    n_no_match  = int((df_pt["reason"] == "no_match").sum())

    content_rate = n_matched / n_content if n_content else float("nan")
    overall_rate = n_matched / n_total   if n_total  else float("nan")

    print(
        f"{pt}: {n_matched}/{n_content} content matched ({content_rate:.1%})  "
        f"| overall {n_matched}/{n_total} ({overall_rate:.1%})  "
        f"[too-short={n_too_short}, filler-only={n_filler}, no-match={n_no_match}]"
    )

YEY: 610/690 content matched (88.4%)  | overall 610/827 (73.8%)  [too-short=133, filler-only=4, no-match=80]
YEZ: 599/662 content matched (90.5%)  | overall 599/929 (64.5%)  [too-short=260, filler-only=7, no-match=63]
YFA: 780/959 content matched (81.3%)  | overall 780/1129 (69.1%)  [too-short=163, filler-only=7, no-match=179]
YFC: 355/431 content matched (82.4%)  | overall 355/518 (68.5%)  [too-short=81, filler-only=6, no-match=76]
YFF: 611/686 content matched (89.1%)  | overall 611/901 (67.8%)  [too-short=201, filler-only=14, no-match=75]
YFG: 243/264 content matched (92.0%)  | overall 243/343 (70.8%)  [too-short=72, filler-only=7, no-match=21]
YFI: 386/426 content matched (90.6%)  | overall 386/476 (81.1%)  [too-short=50, filler-only=0, no-match=40]
YFK: 414/452 content matched (91.6%)  | overall 414/539 (76.8%)  [too-short=87, filler-only=0, no-match=38]
YFM: 840/887 content matched (94.7%)  | overall 840/1249 (67.3%)  [too-short=353, filler-only=9, no-match=47]
YFN: 335/444 conten

## 9 — Save intermediate CSVs

In [11]:
for pt, info in patients.items():
    out_dir: Path = info["out_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)
    pt_matches[pt].to_csv(out_dir / "matches.csv")
    pt_timings[pt].to_csv(out_dir / "timings.csv")
    print(f"{pt}: saved → {out_dir}")

YEY: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YEY/accuracy


YEZ: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YEZ/accuracy
YFA: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFA/accuracy
YFC: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFC/accuracy
YFF: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFF/accuracy
YFG: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFG/accuracy
YFI: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFI/accuracy
YFK: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFK/accuracy
YFM: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFM/accuracy
YFN: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFN/accuracy
YFP: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFP/accuracy
YFR: saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/YFR/accuracy
YFS: saved → /mnt/labworlds/Hayden/Hayden_L

## 10 — SBERT cosine similarity via SLURM

Submits one GPU job per patient.  Re-running is idempotent — patients with an existing `matches_sbert.csv` are skipped.  Set `DRY_RUN = True` to preview sbatch scripts without submitting.

In [12]:
SLURM_LOGS_ROOT    = CONVO_COMP_ROOT / "sbert_slurm_logs"
SLURM_SCRIPTS_ROOT = CONVO_COMP_ROOT / "sbert_slurm_scripts"
SLURM_LOGS_ROOT.mkdir(parents=True, exist_ok=True)
SLURM_SCRIPTS_ROOT.mkdir(parents=True, exist_ok=True)

DRY_RUN = False

submitted: list[tuple] = []
skipped:   list[str]   = []

for pt, info in patients.items():
    input_csv  = info["out_dir"] / "matches.csv"
    output_csv = info["out_dir"] / "matches_sbert.csv"

    if output_csv.exists():
        skipped.append(pt)
        print(f"skip {pt}: {output_csv.name} already exists")
        continue
    if not input_csv.exists():
        print(f"skip {pt}: matches.csv missing — run matching step first")
        continue

    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=sbert_{pt}
#SBATCH --partition={PARTITION}
#SBATCH --gres={GRES}
#SBATCH --cpus-per-task=4
#SBATCH --mem={MEM}
#SBATCH --qos=default_tier
#SBATCH --time=01:00:00
#SBATCH --output={SLURM_LOGS_ROOT}/{pt}_%j.out
#SBATCH --error={SLURM_LOGS_ROOT}/{pt}_%j.err

set -eo pipefail

{CONDA_ACTIVATE}

export HF_HOME='{HF_HOME}'
export TRANSFORMERS_OFFLINE=1
export HF_HUB_OFFLINE=1

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {pt}"

{PYTHON_BIN} {WORKER_PATH} \\
    --input_csv  "{input_csv}" \\
    --output_csv "{output_csv}"

echo "END: $(date)"
"""
    sbatch_path = SLURM_SCRIPTS_ROOT / f"{pt}_sbert.sbatch"
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((pt, "dry-run"))
        print(f"dry-run: {pt}  →  {sbatch_path}")
    else:
        try:
            res = subprocess.run(
                ["sbatch", str(sbatch_path)],
                capture_output=True, text=True, check=True,
            )
            submitted.append((pt, res.stdout.strip()))
            print(f"submitted: {pt}  →  {res.stdout.strip()}")
        except subprocess.CalledProcessError as exc:
            print(f"FAILED SUBMISSION: {pt}\n{exc.stderr}")

print(f"\nsubmitted={len(submitted)}, skipped={len(skipped)}")

skip YEY: matches_sbert.csv already exists
skip YEZ: matches_sbert.csv already exists
skip YFA: matches_sbert.csv already exists
skip YFC: matches_sbert.csv already exists
skip YFF: matches_sbert.csv already exists
skip YFG: matches_sbert.csv already exists
skip YFI: matches_sbert.csv already exists
skip YFK: matches_sbert.csv already exists
skip YFM: matches_sbert.csv already exists
skip YFN: matches_sbert.csv already exists
skip YFP: matches_sbert.csv already exists
skip YFR: matches_sbert.csv already exists
skip YFS: matches_sbert.csv already exists
skip YFU: matches_sbert.csv already exists
skip YFV: matches_sbert.csv already exists

submitted=0, skipped=15


## 11 — Collect results (re-run after SLURM jobs finish)

Patients with `matches_sbert.csv` are loaded with `cosine_sim`.  Pending patients are loaded without it (column is `NaN`).

In [13]:
pt_matches_final: dict[str, pd.DataFrame] = {}
pt_timings_final: dict[str, pd.DataFrame] = {}
sbert_done:    list[str] = []
sbert_pending: list[str] = []

for pt, info in patients.items():
    sbert_csv   = info["out_dir"] / "matches_sbert.csv"
    matches_csv = info["out_dir"] / "matches.csv"
    timings_csv = info["out_dir"] / "timings.csv"

    if sbert_csv.exists():
        df = pd.read_csv(sbert_csv, index_col=0)
        sbert_done.append(pt)
    elif matches_csv.exists():
        df = pd.read_csv(matches_csv, index_col=0)
        df["cosine_sim"] = np.nan
        sbert_pending.append(pt)
    else:
        continue

    df["patient"] = pt
    pt_matches_final[pt] = df

    if timings_csv.exists():
        td = pd.read_csv(timings_csv, index_col=0)
        td["patient"] = pt
        pt_timings_final[pt] = td

print(f"SBERT complete : {sbert_done}")
print(f"SBERT pending  : {sbert_pending}")

SBERT complete : ['YEY', 'YEZ', 'YFA', 'YFC', 'YFF', 'YFG', 'YFI', 'YFK', 'YFM', 'YFN', 'YFP', 'YFR', 'YFS', 'YFU', 'YFV']
SBERT pending  : []


## 12 — Summary statistics and combined save

In [14]:
all_matches = pd.concat(pt_matches_final.values(), ignore_index=True)
all_timings = (
    pd.concat(pt_timings_final.values(), ignore_index=True)
    if pt_timings_final else pd.DataFrame()
)

summary = (
    all_matches.groupby("patient")
    .agg(
        n_sentences    = ("matched", "count"),
        n_matched      = ("matched", "sum"),
        match_rate     = ("matched", "mean"),
        median_wer_sim = ("wer_sim",    "median"),
        median_cos_sim = ("cosine_sim", "median"),
    )
    .reset_index()
)
print(summary.to_string(index=False))

patient  n_sentences  n_matched  match_rate  median_wer_sim  median_cos_sim
    YEY          827        610    0.737606        0.942810        0.972012
    YEZ          929        599    0.644779        0.928571        0.969840
    YFA         1129        780    0.690877        0.857143        0.922784
    YFC          518        355    0.685328        0.941176        0.981028
    YFF          901        611    0.678135        0.875000        0.933114
    YFG          343        243    0.708455        0.928571        0.982148
    YFI          476        386    0.810924        0.916667        0.975532
    YFK          539        414    0.768089        1.000000        1.000000
    YFM         1249        840    0.672538        0.933333        0.975732
    YFN          587        335    0.570698        0.888889        0.929464
    YFP          909        760    0.836084        0.944444        0.986475
    YFR          836        658    0.787081        1.000000        1.000000
    YFS     

In [15]:
if not all_timings.empty:
    abs_err = all_timings["delta_mid_rel_s"].abs()
    print("Word-level drift  (within-sentence relative midpoint, seconds):")
    print(f"  median |err| : {abs_err.median():.3f} s")
    print(f"  mean   |err| : {abs_err.mean():.3f} s")
    print(f"  95th %%      : {abs_err.quantile(0.95):.3f} s")

Word-level drift  (within-sentence relative midpoint, seconds):
  median |err| : 0.073 s
  mean   |err| : 0.365 s
  95th %%      : 1.489 s


## 13 — Drift statistics: segment-level and word-level

**Segment-level drift** (`seg_mid_drift_s`): absolute timing offset of the matched sentence pair,
`hyp_sentence_midpoint − ref_sentence_midpoint`.  Captures how far WhisperX timestamps the
whole sentence from the Praat reference (positive = hyp is later).

**Word-level drift** (`delta_mid_rel_s`): within-sentence relative timing error per matched word
pair — anchored to the sentence start so global offset cancels.  Captures how tightly individual
words are aligned inside a sentence.

In [16]:
matched_df = all_matches[all_matches["matched"]].copy()

# ── Segment-level drift (absolute) ──────────────────────────────────────────
seg_drift     = matched_df["seg_mid_drift_s"].dropna()
seg_drift_abs = seg_drift.abs()

print("Segment-level drift  (hyp_sentence_mid − ref_sentence_mid, seconds):")
print(f"  median |drift| : {seg_drift_abs.median():.3f} s")
print(f"  mean   |drift| : {seg_drift_abs.mean():.3f} s")
print(f"  95th %%        : {seg_drift_abs.quantile(0.95):.3f} s")
print(f"  signed mean    : {seg_drift.mean():.3f} s   (+ = hyp systematically later)")

# ── Word-level drift (within-sentence relative) ──────────────────────────────
if not all_timings.empty:
    word_abs = all_timings["delta_mid_rel_s"].abs()
    print("\nWord-level drift  (within-sentence relative midpoint, seconds):")
    print(f"  median |drift| : {word_abs.median():.3f} s")
    print(f"  mean   |drift| : {word_abs.mean():.3f} s")
    print(f"  95th %%        : {word_abs.quantile(0.95):.3f} s")

# ── Per-patient breakdown ────────────────────────────────────────────────────
print("\nPer-patient segment-level drift:")
pt_seg = (
    matched_df.groupby("patient")["seg_mid_drift_s"]
    .agg(
        n_sents        = "count",
        median_abs_s   = lambda x: x.abs().median(),
        mean_abs_s     = lambda x: x.abs().mean(),
        signed_mean_s  = "mean",
        p95_abs_s      = lambda x: x.abs().quantile(0.95),
    )
    .reset_index()
)
print(pt_seg.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

if not all_timings.empty:
    print("\nPer-patient word-level drift:")
    pt_word = (
        all_timings.groupby("patient")["delta_mid_rel_s"]
        .agg(
            n_words        = "count",
            median_abs_s   = lambda x: x.abs().median(),
            mean_abs_s     = lambda x: x.abs().mean(),
            p95_abs_s      = lambda x: x.abs().quantile(0.95),
        )
        .reset_index()
    )
    print(pt_word.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

Segment-level drift  (hyp_sentence_mid − ref_sentence_mid, seconds):
  median |drift| : 0.116 s
  mean   |drift| : 1.191 s
  95th %%        : 6.089 s
  signed mean    : -0.800 s   (+ = hyp systematically later)

Word-level drift  (within-sentence relative midpoint, seconds):
  median |drift| : 0.073 s
  mean   |drift| : 0.365 s
  95th %%        : 1.489 s

Per-patient segment-level drift:
patient  n_sents  median_abs_s  mean_abs_s  signed_mean_s  p95_abs_s
    YEY        0           NaN         NaN            NaN        NaN
    YEZ        0           NaN         NaN            NaN        NaN
    YFA        0           NaN         NaN            NaN        NaN
    YFC        0           NaN         NaN            NaN        NaN
    YFF        0           NaN         NaN            NaN        NaN
    YFG        0           NaN         NaN            NaN        NaN
    YFI        0           NaN         NaN            NaN        NaN
    YFK      414         0.052       0.809         -0.610

/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of

In [17]:
combined_out = CONVO_COMP_ROOT / "combined_accuracy"
combined_out.mkdir(parents=True, exist_ok=True)

all_matches.to_csv(combined_out / "all_matches.csv", index=False)
if not all_timings.empty:
    all_timings.to_csv(combined_out / "all_timings.csv", index=False)

print(f"saved → {combined_out}")
print(f"  all_matches.csv : {len(all_matches)} rows  (patients: {sorted(all_matches['patient'].unique())})")
if not all_timings.empty:
    print(f"  all_timings.csv : {len(all_timings)} rows")

saved → /mnt/labworlds/Hayden/Hayden_Lab/speech_247/convo_comparison/combined_accuracy
  all_matches.csv : 12928 rows  (patients: ['YEY', 'YEZ', 'YFA', 'YFC', 'YFF', 'YFG', 'YFI', 'YFK', 'YFM', 'YFN', 'YFP', 'YFR', 'YFS', 'YFU', 'YFV'])
  all_timings.csv : 82245 rows


In [18]:
# Sample matched sentences for a quick sanity check
pt_example = sorted(pt_matches_final.keys())[0]
matched_ex = pt_matches_final[pt_example]
matched_ex = matched_ex[matched_ex["matched"] == True].head(8)

for _, row in matched_ex.iterrows():
    print(f"\033[1mref:\033[0m {row.ref_text}")
    print(f"\033[1mhyp:\033[0m {row.hyp_text}")
    wer_s = f"WER-sim={row.wer_sim:.2f}"
    cos_s = (f"cos-sim={row.cosine_sim:.2f}" if pd.notna(row.get("cosine_sim")) else "cos-sim=pending")
    print(f"  [{wer_s}  {cos_s}]")
    print()

ref: yeah take whatever you want
hyp: yeah take whatever you want
  [WER-sim=1.00  cos-sim=1.00]

ref: the milk and the the you wanna take the yogurt
hyp: the milk and the do you want to take the yogurts
  [WER-sim=0.70  cos-sim=0.93]

ref: milk and yogurt
hyp: milk and
  [WER-sim=0.67  cos-sim=0.83]

ref: oh lord
hyp: ooh lord
  [WER-sim=1.00  cos-sim=0.79]

ref: laughs laughs i was 'bout to go home wasn't telling my supervisor
hyp: warm okay i was about to go home wasn't telling my supervisor
  [WER-sim=0.83  cos-sim=0.71]

ref: she told me she said name please
hyp: she told me she said lonnie please
  [WER-sim=0.86  cos-sim=0.71]

ref: i said get me off of this floor
hyp: i said get me off of this float
  [WER-sim=1.00  cos-sim=0.75]

ref: it was on another floor
hyp: it was on another float
  [WER-sim=1.00  cos-sim=0.56]

